In [ ]:
# Weekly Project 4
import glob

import cv2
import matplotlib.pyplot as plt
import numpy as np
#First we simply collect all the checkerboard points from each camera for calibration. Our approach is to first calibrate each camera separately, as is happening in the following two cells
nb_vertical = 6
nb_horizontal = 9

# Prepare object points, like (0, 0, 0), (1, 0, 0), (2, 0, 0) ....,(6, 5, 0).
objp = np.zeros((nb_horizontal * nb_vertical, 3), np.float32)
objp[:, :2] = np.mgrid[0:nb_vertical, 0:nb_horizontal].T.reshape(-1, 2)

# Arrays to store object points and image points from all the images.
objpoints_left = []  # 3D point in real world space.
objpoints_right = []  # 3D point in real world space.
imgpoints_left = []  # 2D points in image plane.
imgpoints_right = []  # 2D points in image plane.

imageleft = glob.glob("C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\week4\\myn\\left-*.png")
imageleft.sort()
assert imageleft

for fname in imageleft:
    img_left = cv2.imread(fname)
    img_right = cv2.imread(fname.replace("left", "right"))
    h, w = img_left.shape[:2]
    gray_left = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)

    ret_left, corners_left = cv2.findChessboardCorners(
        gray_left, (nb_vertical, nb_horizontal), None
    )
    ret_right, corners_right = cv2.findChessboardCorners(
        gray_right, (nb_vertical, nb_horizontal), None
    )

    # Display both images with the detected corners.


    # If found, add object points and image points (after refining them).
    if ret_left and ret_right:
        display_left = img_left.copy()
        display_right = img_right.copy()
        cv2.drawChessboardCorners(
            display_left, (nb_vertical, nb_horizontal), corners_left, ret_left
        )
        cv2.drawChessboardCorners(
            display_right, (nb_vertical, nb_horizontal), corners_right, ret_right
        )
        cv2.imshow("Left Image", display_left)
        cv2.imshow("Right Image", display_right)
        cv2.waitKey(50)
        objpoints_left.append(objp)
        objpoints_right.append(objp)

        imgpoints_left.append(corners_left)
        imgpoints_right.append(corners_right)

cv2.destroyAllWindows()
ret_left, mtx_left, dist_left, rvecs_left, tvecs_left = cv2.calibrateCamera(
    objpoints_left, imgpoints_left, gray_left.shape[::-1], None, None
)
ret_right, mtx_right, dist_right, rvecs_right, tvecs_right = cv2.calibrateCamera(
    objpoints_right, imgpoints_right, gray_right.shape[::-1], None, None
)
img_left = cv2.imread("C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\week4\\myn\\left-0001.png")
img_right = cv2.imread("C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\week4\\myn\\right-0001.png")
h, w = img_left.shape[:2]
newcameramtx_right, roi_right = cv2.getOptimalNewCameraMatrix(mtx_right, dist_right, (w, h), 1, (w, h))
newcameramtx_left, roi_left = cv2.getOptimalNewCameraMatrix(mtx_left, dist_left, (w, h), 1, (w, h))
# Observe the refined camera matrix computed for the target image size.
print("New camera matrix left:")
print(newcameramtx_left)
print("New camera matrix right:")
print(newcameramtx_right)

# Please see the raw camera matrix returned directly by calibrateCamera(). The two
# matrices are related but not identical. This one describes the camera's
# intrinsic geometry in general, while the refined version above is adjusted
# for the chosen output image size/cropping behavior.
print("Camera matrix left:")
print(mtx_left)
print("Camera matrix right:")
print(mtx_right)

# Inspect the lens distortion coefficients found during calibration. This set
# of numbers captures how much the lens bends light. Primarily radial distortion
# (things appear to bulge or pinch, especially near the image edges; "barrel"
# or "pincushion" distortion) and a smaller tangential distortion (from the
# lens not being perfectly parallel to the image sensor).
print("Distortion coefficients left:")
print(dist_left)
print("Distortion coefficients right:")
print(dist_right)
# Since we have our camera matrix and distortion coefficients, we are going to undistort the images
undistorted_imgs_left = []
undistorted_imgs_right = []
for fname in imageleft:
    img_left = cv2.imread(fname)
    img_right = cv2.imread(fname.replace("left", "right"))
    dst = cv2.undistort(img_left, mtx_left, dist_left, None, None)
    undistorted_imgs_left.append(dst)
    dst = cv2.undistort(img_right, mtx_right, dist_right, None, None)
    undistorted_imgs_right.append(dst)
# Compare distorted and undistorted images from the same stereo pair.
left_distorted = cv2.imread(imageleft[0])
right_distorted = cv2.imread(imageleft[0].replace("left", "right"))
left_undistorted = cv2.undistort(
    left_distorted, mtx_left, dist_left, None, newcameramtx_left
)
right_undistorted = cv2.undistort(
    right_distorted, mtx_right, dist_right, None, newcameramtx_right
)
x_l, y_l, w_l, h_l = roi_left
print(x_l,y_l,w_l,h_l)
left_undistorted = left_undistorted[y_l : y_l + h_l, x_l : x_l + w_l]

x_r, y_r, w_r, h_r = roi_right
right_undistorted = right_undistorted[y_r : y_r + h_r, x_r : x_r + w_r]

images = [
    left_distorted,
    left_undistorted,
    right_distorted,
    right_undistorted,
]
titles = [
    "Left - distorted",
    "Left - undistorted",
    "Right - distorted",
    "Right - undistorted",
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for axis, image, title in zip(axes.ravel(), images, titles):
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()
Now that we have calibrated the cameras individually, we can utilize the following information: focal length and optical center from mtx, lens distortion from dist, known physical checkerboard points in objpoints and the same points on the image plane in imgpoints.

What we need to find now is the two cameras positions relative to each other.
We use the StereoCalibrate function from cv2 which is an iterative optimisation function. This will repeat until it meets some criteria, hence we set some criteria.
This function returns the R and T matrix which specifies the rotation and translation between the two cameras (L to R camera). We then use stereoRectify to calculate how each camera view should be rotated and implement those changes using initUnitDistortRectifyMap on the image.
flags = cv2.CALIB_FIX_INTRINSIC
criteria_stereo = (
    cv2.TERM_CRITERIA_MAX_ITER + cv2.TERM_CRITERIA_EPS,
    100,
    1e-5,
)

ret_stereo, _, _, _, _, R, T, E, F = cv2.stereoCalibrate(
    objpoints_left,
    imgpoints_left,
    imgpoints_right,
    mtx_left,
    dist_left,
    mtx_right,
    dist_right,
    (w, h),
    criteria=criteria_stereo,
    flags=flags,
)

R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
    mtx_left,
    dist_left,
    mtx_right,
    dist_right,
    (w, h),
    R,
    T,
    flags=cv2.CALIB_ZERO_DISPARITY,
    alpha=0,
)

# Rectification combines lens undistortion and stereo alignment.
map1_left, map2_left = cv2.initUndistortRectifyMap(
    mtx_left, dist_left, R1, P1, (w, h), cv2.CV_32FC1
)
map1_right, map2_right = cv2.initUndistortRectifyMap(
    mtx_right, dist_right, R2, P2, (w, h), cv2.CV_32FC1
)
for fname in imageleft:
    img_left = cv2.imread(fname)
    img_right = cv2.imread(fname.replace("left", "right"))
    rectified_left = cv2.remap(img_left, map1_left, map2_left, cv2.INTER_LINEAR)
    rectified_right = cv2.remap(img_right, map1_right, map2_right, cv2.INTER_LINEAR)
    before_left = img_left.copy()
    before_right = img_right.copy()
    after_left = rectified_left.copy()
    after_right = rectified_right.copy()

    for y in range(0, h, 30):
        cv2.line(before_left, (0, y), (w, y), (0, 0, 255), 1)
        cv2.line(before_right, (0, y), (w, y), (0, 0, 255), 1)
        cv2.line(after_left, (0, y), (w, y), (0, 0, 255), 1)
        cv2.line(after_right, (0, y), (w, y), (0, 0, 255), 1)

    before = np.hstack((before_left, before_right))
    after = np.hstack((after_left, after_right))

    cv2.imshow("Before rectification", before)
    cv2.imshow("After rectification", after)
    cv2.waitKey(1)
cv2.destroyAllWindows()
def row_absolute_difference(gray1, gray2):
    output = np.zeros_like(gray1)
    [h, w] = gray1.shape
    for i in range(h):
        for j in range(w):
            output[i, j] = abs(int(gray1[i, j]) - int(gray2[i, j]))
    return output, np.mean(output, dtype=np.float64)
Now that we have our stereo rectified images, we run and display our disparity map like we did last week.
threshold = 15  # Threshold for early exit in disparity calculation
limit = 5
iwannagohome = 0
for fname in imageleft:
    # Compute the disparity heatmap for one stereo pair.
    img_left = cv2.imread(fname)
    img_right = cv2.imread(fname.replace("left", "right"))
    rectified_left = cv2.remap(img_left, map1_left, map2_left, cv2.INTER_LINEAR)
    rectified_right = cv2.remap(img_right, map1_right, map2_right, cv2.INTER_LINEAR)

    print(roi_left)
    print(roi_right)

    # x_l, y_l, w_l, h_l = roi_left
    # rectified_left = rectified_left[y_l : y_l + h_l, x_l : x_l + w_l]
    # x_r, y_r, w_r, h_r = roi_right
    # rectified_right = rectified_right[y_r : y_r + h_r, x_r : x_r + w_r]

    # Use integer dimensions; cv2.resize expects (width, height).
    org_size = rectified_left.shape[:2]  # (height, width)
    resize_size = (
        max(1, org_size[1] // 15),
        max(1, org_size[0] // 15),
    )
    rectified_left = cv2.resize(
        rectified_left, resize_size, interpolation=cv2.INTER_AREA
    )
    rectified_right = cv2.resize(
        rectified_right, resize_size, interpolation=cv2.INTER_AREA
    )
    gray_left = cv2.cvtColor(rectified_left, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(rectified_right, cv2.COLOR_BGR2GRAY)

    [box_height, box_width] = [5, 5]
    [img_height, img_width] = gray_left.shape
    distance = np.zeros(
        (img_height - box_height + 1, img_width - box_width + 1),
        dtype=np.float64,
    )

    for i in range(0, img_height - box_height + 1):
        for j in range(0, img_width - box_width + 1):
            lowestdiff = float("inf")
            for jj in range(0, img_width - box_width + 1):
                if lowestdiff < threshold:
                    break
                _, total_diff = row_absolute_difference(
                    gray_left[i : i + box_height, j : j + box_width],
                    gray_right[i : i + box_height, jj : jj + box_width],
                )
                if total_diff < lowestdiff:
                    lowestdiff = total_diff
                    best_position = jj

            if lowestdiff < 15:
                distance[i, j] = abs(best_position - j)

    # Normalize disparity and display it as a colored heatmap.
    distance_scaled = cv2.normalize(
        distance, None, 0, 255, cv2.NORM_MINMAX
    ).astype(np.uint8)
    distance_scaled[distance_scaled > 100] = 0
    distance_scaled = np.clip(
        distance_scaled.astype(np.float32) * 2.5, 0, 255
    ).astype(np.uint8)

    # Resize back to the original size for display.
    distance_scaled = cv2.resize(
        distance_scaled, (org_size[1], org_size[0]), interpolation=cv2.INTER_AREA
    )
    rectified_left = cv2.remap(img_left, map1_left, map2_left, cv2.INTER_LINEAR)

    disparity_heatmap = cv2.applyColorMap(distance_scaled, cv2.COLORMAP_JET)
    bothim = np.hstack((disparity_heatmap, rectified_left))
    cv2.imshow("Disparity Heatmap", bothim)
    cv2.waitKey(0)

cv2.destroyAllWindows()

